# 2020 vs 2024 Presidential Election Comparison

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 16,
        "figure.figsize": (12, 8),
        "figure.dpi": 200,
        "figure.edgecolor": "k",
        "figure.facecolor": "w",
    }
)

In [ ]:
# Load data
try:
    df_20 = pd.read_feather("2020.ft")
    df_24 = pd.read_feather("2024.ft")
    print("Loaded both 2020 and 2024 data successfully")
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    print("Please ensure both 2020.ft and 2024.ft exist")
    df_20 = None
    df_24 = None

In [ ]:
if df_20 is not None and df_24 is not None:
    # Merge datasets on county
    df_comparison = df_20.merge(df_24, on="county", how="inner")
    print(f"Comparison dataset: {len(df_comparison)} counties")
    
    # Identify candidate columns
    # 2020 columns
    biden_col_20 = [col for col in df_20.columns if "biden" in col.lower() and "pct" in col][0]
    trump_col_20 = [col for col in df_20.columns if "trump" in col.lower() and "pct" in col][0]
    
    # 2024 columns
    harris_col_24 = [col for col in df_24.columns if "harris" in col.lower() and "pct" in col][0]
    trump_col_24 = [col for col in df_24.columns if "trump" in col.lower() and "pct" in col][0]
    
    print(f"2020 Biden column: {biden_col_20}")
    print(f"2020 Trump column: {trump_col_20}")
    print(f"2024 Harris column: {harris_col_24}")
    print(f"2024 Trump column: {trump_col_24}")
    
    # Calculate changes
    df_comparison["trump_change"] = df_comparison[trump_col_24] - df_comparison[trump_col_20]
    df_comparison["dem_change"] = df_comparison[harris_col_24] - df_comparison[biden_col_20]
    df_comparison["turnout_change"] = (df_comparison["pres_24_total"] - df_comparison["pres_20_total"]) / df_comparison["pres_20_total"]
    
    # Summary statistics
    print("\n" + "="*60)
    print("2020 vs 2024 Presidential Election Comparison")
    print("="*60)
    
    # Statewide totals
    total_20 = df_comparison["pres_20_total"].sum()
    total_24 = df_comparison["pres_24_total"].sum()
    turnout_change_pct = ((total_24 - total_20) / total_20) * 100
    
    print(f"\nStatewide Results:")
    print(f"2020 Total Votes: {total_20:,}")
    print(f"2024 Total Votes: {total_24:,}")
    print(f"Turnout Change: {turnout_change_pct:+.1f}%")
    
    # Candidate performance changes
    trump_20_total = df_comparison[trump_col_20.replace("_pct", "")].sum()
    trump_24_total = df_comparison[trump_col_24.replace("_pct", "")].sum()
    trump_change_pct = ((trump_24_total - trump_20_total) / trump_20_total) * 100
    
    biden_20_total = df_comparison[biden_col_20.replace("_pct", "")].sum()
    harris_24_total = df_comparison[harris_col_24.replace("pct", "")].sum()
    dem_change_pct = ((harris_24_total - biden_20_total) / biden_20_total) * 100
    
    print(f"\nCandidate Performance Changes:")
    print(f"Trump: {trump_20_total:,} → {trump_24_total:,} ({trump_change_pct:+.1f}%)")
    print(f"Democratic: {biden_20_total:,} → {harris_24_total:,} ({dem_change_pct:+.1f}%)")
    
    # County-level analysis
    print(f"\nCounty-level Changes:")
    print(f"Counties with Trump gains: {(df_comparison['trump_change'] > 0).sum()}")
    print(f"Counties with Democratic gains: {(df_comparison['dem_change'] > 0).sum()}")
    print(f"Counties with turnout increases: {(df_comparison['turnout_change'] > 0).sum()}")
    
    # Top counties by change
    print(f"\nTop 5 counties by Trump gain:")
    top_trump_gain = df_comparison.nlargest(5, "trump_change")[["county", "trump_change"]]
    for _, row in top_trump_gain.iterrows():
        print(f"{row['county']}: +{row['trump_change']*100:.1f} percentage points")
    
    print(f"\nTop 5 counties by Democratic gain:")
    top_dem_gain = df_comparison.nlargest(5, "dem_change")[["county", "dem_change"]]
    for _, row in top_dem_gain.iterrows():
        print(f"{row['county']}: +{row['dem_change']*100:.1f} percentage points")
    
    # Visualizations
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Trump support change distribution
    axes[0, 0].hist(df_comparison["trump_change"], bins=20, alpha=0.7, color='red', edgecolor='black')
    axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.7)
    axes[0, 0].set_xlabel("Trump Support Change (2024 - 2020)")
    axes[0, 0].set_ylabel("Number of Counties")
    axes[0, 0].set_title("Distribution of Trump Support Changes")
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Democratic support change distribution
    axes[0, 1].hist(df_comparison["dem_change"], bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 1].axvline(0, color='black', linestyle='--', alpha=0.7)
    axes[0, 1].set_xlabel("Democratic Support Change (2024 - 2020)")
    axes[0, 0].set_ylabel("Number of Counties")
    axes[0, 1].set_title("Distribution of Democratic Support Changes")
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Turnout change distribution
    axes[0, 2].hist(df_comparison["turnout_change"], bins=20, alpha=0.7, color='green', edgecolor='black')
    axes[0, 2].axvline(0, color='black', linestyle='--', alpha=0.7)
    axes[0, 2].set_xlabel("Turnout Change (2024 - 2020)")
    axes[0, 2].set_ylabel("Number of Counties")
    axes[0, 2].set_title("Distribution of Turnout Changes")
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Trump change vs Democratic change
    axes[1, 0].scatter(df_comparison["trump_change"], df_comparison["dem_change"], alpha=0.6)
    axes[1, 0].axhline(0, color='black', linestyle='--', alpha=0.7)
    axes[1, 0].axvline(0, color='black', linestyle='--', alpha=0.7)
    axes[1, 0].set_xlabel("Trump Support Change")
    axes[1, 0].set_ylabel("Democratic Support Change")
    axes[1, 0].set_ylabel("Democratic Support Change")
    axes[1, 0].set_title("Trump vs Democratic Support Changes")
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 5: 2020 vs 2024 Trump support
    axes[1, 1].scatter(df_comparison[trump_col_20], df_comparison[trump_col_24], alpha=0.6, color='red')
    max_trump = max(df_comparison[trump_col_20].max(), df_comparison[trump_col_24].max())
    axes[1, 1].plot([0, max_trump], [0, max_trump], 'k--', alpha=0.5)
    axes[1, 1].set_xlabel("2020 Trump Support (%)")
    axes[1, 1].set_ylabel("2024 Trump Support (%)")
    axes[1, 1].set_title("2020 vs 2024 Trump Support by County")
    axes[1, 1].grid(True, alpha=0.3)
    
    # Plot 6: 2020 vs 2024 Democratic support
    axes[1, 2].scatter(df_comparison[biden_col_20], df_comparison[harris_col_24], alpha=0.6, color='blue')
    max_dem = max(df_comparison[biden_col_20].max(), df_comparison[harris_col_24].max())
    axes[1, 2].plot([0, max_dem], [0, max_dem], 'k--', alpha=0.5)
    axes[1, 2].set_xlabel("2020 Biden Support (%)")
    axes[1, 2].set_ylabel("2024 Harris Support (%)")
    axes[1, 2].max()
    axes[1, 2].set_title("2020 vs 2024 Democratic Support by County")
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation analysis
    print(f"\nCorrelation Analysis:")
    print("=" * 40)
    
    change_cols = ["trump_change", "dem_change", "turnout_change"]
    correlation_matrix = df_comparison[change_cols].corr()
    print(correlation_matrix)
    
    # Regional analysis by county size
    print(f"\nRegional Analysis by County Size:")
    print("=" * 40)
    
    # Group counties by 2020 size
    df_comparison["county_size_20"] = pd.cut(df_comparison["pres_20_total"], 
                                            bins=3, 
                                            labels=["Small", "Medium", "Large"])
    
    size_analysis = df_comparison.groupby("county_size_20").agg({
        "trump_change": ["mean", "std"],
        "dem_change": ["mean", "std"],
        "turnout_change": ["mean", "std"]
    }).round(3)
    
    print(size_analysis)
    
    # Save comparison results
    comparison_summary = {
        "total_votes_2020": total_20,
        "total_votes_2024": total_24,
        "turnout_change_pct": turnout_change_pct,
        "trump_change_pct": trump_change_pct,
        "dem_change_pct": dem_change_pct,
        "counties_analyzed": len(df_comparison),
        "correlation_trump_dem_changes": df_comparison["trump_change"].corr(df_comparison["dem_change"])
    }
    
    print(f"\nComparison Summary:")
    print("=" * 40)
    for key, value in comparison_summary.items():
        if isinstance(value, float):
            print(f"{key}: {value:.3f}")
    else:
            print(f"{key}: {value:,}")